In [15]:
import pandas as pd
# Cambia 'archivo.csv' por el nombre real de tus archivos
df = pd.read_csv('registros_titulos.csv')
print(df.head())


   anio  mes documento                        nombre_completo  carrera_id  \
0     1   11   4293889              BERNARDINA ROLON GUERRERO         111   
1     1   12   4233874              EDI ISABEL BERNAL BENITEZ         111   
2     2    3   1884029         LIDUVINA CABRERA DE PRESENTADO          41   
3     2   12   1262724                     AIDA MABEL CABRERA          49   
4     3   12   3501374  JORGELINA GOMEZ DE LA FUENTE SANABRIA          79   

                    carrera  titulo_id  \
0                ENFERMERIA        110   
1                ENFERMERIA        792   
2        CIENCIAS CONTABLES        792   
3  CIENCIAS DE LA EDUCACION        580   
4                   DERECHO         62   

                                     titulo numero_resolucion  \
0                LICENCIADO/A EN ENFERMERIA              1545   
1                              LICENCIADO/A               S/R   
2                              LICENCIADO/A       DGES Nº 166   
3  LICENCIADO/A EN CIENC

In [20]:
print(df.isnull().sum())


anio                      0
mes                       0
documento                 0
nombre_completo           0
carrera_id                0
carrera                   0
titulo_id                 0
titulo                    0
numero_resolucion         3
fecha_resolucion         14
tipo_institucion_id      28
tipo_institucion         28
institucion_id            0
institucion               0
gobierno_actual           0
sexo                   1163
dtype: int64


**Función de limpieza categórica avanzada que incluye:**

    Eliminación de espacios al inicio y final.
    Reemplazo de dobles (o múltiples) espacios internos por uno solo.
    Normalización de acentos.
    Eliminación de caracteres especiales (opcional).
    Conversión a minúsculas.

In [18]:
import pandas as pd
import unicodedata
import re

def limpiar_texto_avanzado(s):
    if pd.isnull(s):
        return s
    # Convertir a string, minúsculas y eliminar espacios al inicio/final
    s = str(s).strip().lower()
    # Normalizar acentos
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    # Reemplazar múltiples espacios internos por uno solo
    s = re.sub(r'\s+', ' ', s)
    # Eliminar caracteres especiales (mantener solo letras, números y espacios)
    s = re.sub(r'[^a-z0-9áéíóúüñ ]', '', s)
    return s

# Copia del DataFrame original
df_limpio = df.copy()

# Aplica limpieza a todas las columnas categóricas
for col in df_limpio.select_dtypes(include=['object']).columns:
    df_limpio[col] = df_limpio[col].apply(limpiar_texto_avanzado)

# Elimina filas duplicadas
df_limpio = df_limpio.drop_duplicates()

# Comparación antes vs después
print("ANTES:")
print(df.select_dtypes(include=['object']).nunique())
print("Filas:", len(df))

print("\nDESPUÉS:")
print(df_limpio.select_dtypes(include=['object']).nunique())
print("Filas:", len(df_limpio))


ANTES:
documento            458959
nombre_completo      458268
carrera                2032
titulo                  963
numero_resolucion    106929
fecha_resolucion       5921
tipo_institucion         12
institucion             513
gobierno_actual           2
sexo                      2
dtype: int64
Filas: 655277

DESPUÉS:
documento            457817
nombre_completo      457055
carrera                1924
titulo                  887
numero_resolucion    106865
fecha_resolucion       5921
tipo_institucion          7
institucion             444
gobierno_actual           2
sexo                      2
dtype: int64
Filas: 655252


**Detección de homonimos**

In [24]:
# Agrupa por nombre y cuenta cuántos documentos distintos tiene cada nombre
homonimos = df_limpio.groupby('nombre_completo')['documento'].nunique()
# Filtra los nombres que aparecen con más de un documento
casos_homonimia = homonimos[homonimos > 1]
print(f"Cantidad de nombres con homonimia: {len(casos_homonimia)}")

# Ordena de mayor a menor y muestra los primeros 10
casos_homonimia_ordenados = casos_homonimia.sort_values(ascending=False)
print("Top 10 nombres con más homonimia:")
print(casos_homonimia_ordenados.head(10))


Cantidad de nombres con homonimia: 1253
Top 10 nombres con más homonimia:
nombre_completo
maria belen gonzalez           4
graciela martinez              4
marcos antonio gonzalez        4
maria liz gonzalez             4
gabriela duarte                3
julio cesar caceres            3
juan carlos benitez benitez    3
julio cesar rodriguez          3
liliana martinez gonzalez      3
juan gabriel benitez           3
Name: documento, dtype: int64


In [23]:
# Agrupa por documento y cuenta cuántos nombres distintos tiene cada documento
doc_duplicados = df_limpio.groupby('documento')['nombre_completo'].nunique()
# Filtra los documentos que aparecen con más de un nombre
casos_doc_duplicados = doc_duplicados[doc_duplicados > 1]
print(f"Cantidad de documentos asociados a más de un nombre: {len(casos_doc_duplicados)}")
print(casos_doc_duplicados.sort_values(ascending=False).head(10))  # Muestra los 10 casos más frecuentes

# Si quieres ver los nombres asociados a un documento específico:
ejemplo_doc = casos_doc_duplicados.index[0]
print(df_limpio[df_limpio['documento'] == ejemplo_doc][['documento', 'nombre_completo']])


Cantidad de documentos asociados a más de un nombre: 538
documento
4697844           3
mg10514205        2
0                 2
000570415la034    2
0030296990        2
0201442612        2
040916181         2
091686584         2
1012740           2
1021004           2
Name: nombre_completo, dtype: int64
       documento       nombre_completo
25436          0     eron paulo favero
169330         0  ydalina smith colman


In [28]:
# Detectar documentos asociados a más de un nombre
doc_duplicados = df_limpio.groupby('documento')['nombre_completo'].nunique()
casos_doc_duplicados = doc_duplicados[doc_duplicados > 1]

# Filtrar todas las filas con esos documentos
df_docs_duplicados = df_limpio[df_limpio['documento'].isin(casos_doc_duplicados.index)][['documento', 'nombre_completo']]

# Agregar columna con cantidad de nombres asociados
df_docs_duplicados['cantidad_nombres'] = df_docs_duplicados['documento'].map(casos_doc_duplicados)

# Ordenar por cantidad de nombres asociados (de mayor a menor)
df_docs_duplicados = df_docs_duplicados.sort_values('cantidad_nombres', ascending=False)

# Exportar a Excel
df_docs_duplicados.to_excel('documentos_duplicados_ordenados.xlsx', index=False)


In [30]:
from difflib import SequenceMatcher

def nombre_similar(n1, n2, threshold=0.85):
    return SequenceMatcher(None, n1, n2).ratio() >= threshold

# Agrupa por documento
resultados = []
for doc, grupo in df_docs_duplicados.groupby('documento'):
    nombres = grupo['nombre_completo'].drop_duplicates().tolist()
    if len(nombres) > 1:
        # Compara todos los pares de nombres
        similares = 0
        distintos = 0
        for i in range(len(nombres)):
            for j in range(i+1, len(nombres)):
                if nombre_similar(nombres[i], nombres[j]):
                    similares += 1
                else:
                    distintos += 1
        resultados.append({
            'documento': doc,
            'nombres': nombres,
            'similares': similares,
            'distintos': distintos,
            'total_nombres': len(nombres)
        })

# Contar casos
casos_misma_persona = sum(1 for r in resultados if r['similares'] > 0 and r['distintos'] == 0)
casos_personas_distintas = sum(1 for r in resultados if r['distintos'] > 0)

print(f"Documentos duplicados que parecen la misma persona (nombre similar): {casos_misma_persona}")
print(f"Documentos duplicados que parecen personas distintas (nombre diferente): {casos_personas_distintas}")


Documentos duplicados que parecen la misma persona (nombre similar): 103
Documentos duplicados que parecen personas distintas (nombre diferente): 435
